<a href="https://colab.research.google.com/github/saadhana192465019/Digital-Forensics-and-cyber-crime-Investigation--CSA6102/blob/main/Experiment42_Section_65B_Certificate_Validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ============================================================
# EXPERIMENT 4
# SECTION 65B CERTIFICATE VALIDATION
# Generating and Validating a Certificate under Section 65B(4)
# ============================================================

import hashlib
from datetime import datetime, timezone, timedelta


# ============================================================
# 1. INDIAN STANDARD TIME (IST)
# ============================================================

IST = timezone(
    timedelta(hours=5, minutes=30)
)


# ============================================================
# 2. REQUIRED FIELDS
# ============================================================

REQUIRED_FIELDS = [

    "case_reference",
    "electronic_record",
    "source_device",

    # Four statutory matters
    "regular_use_statement",
    "ordinary_activity",
    "proper_operation",
    "accuracy_statement",

    # Integrity and certificate details
    "hash_value",
    "deponent_name",
    "deponent_designation",
    "date_of_certificate"
]


# ============================================================
# 3. CUSTOM EXCEPTION
# ============================================================

class CertificateError(Exception):
    pass


# ============================================================
# 4. VALIDATE CERTIFICATE DETAILS
# ============================================================

def validate(details):

    missing = []

    for field in REQUIRED_FIELDS:

        value = str(
            details.get(field, "")
        ).strip()

        if not value:

            missing.append(field)

    return missing


# ============================================================
# 5. GENERATE SECTION 65B(4) CERTIFICATE
# ============================================================

def generate_65b_certificate(
    details,
    strict=True
):

    missing = validate(details)

    # Strict mode refuses incomplete certificates
    if missing and strict:

        raise CertificateError(
            "Certificate incomplete. "
            "Missing fields: "
            + ", ".join(missing)
        )

    d = details

    certificate = f"""
CERTIFICATE UNDER SECTION 65B(4)
OF THE INDIAN EVIDENCE ACT, 1872
============================================================

Case Reference:
{d.get("case_reference")}

Date of Certificate:
{d.get("date_of_certificate")}


1. IDENTIFICATION OF THE ELECTRONIC RECORD
------------------------------------------------------------

{d.get("electronic_record")}

SHA-256 Hash Value:
{d.get("hash_value")}


2. PARTICULARS OF THE DEVICE
------------------------------------------------------------

{d.get("source_device")}


3. STATEMENTS REQUIRED UNDER SECTION 65B(2)
------------------------------------------------------------

(a) Regular Use:
{d.get("regular_use_statement")}


(b) Ordinary Activity:
{d.get("ordinary_activity")}


(c) Proper Operation:
{d.get("proper_operation")}


(d) Accuracy of Information:
{d.get("accuracy_statement")}


4. DECLARATION
------------------------------------------------------------

The above statements are true to the best of my knowledge
and belief. This certificate is given by me in my capacity
as a responsible official occupying the position stated below
in relation to the operation of the relevant device.


Name:
{d.get("deponent_name")}

Designation:
{d.get("deponent_designation")}

Signature:
____________________________


============================================================
END OF CERTIFICATE
============================================================
""".strip()

    return certificate, missing


# ============================================================
# 6. SAMPLE CASE DETAILS
# ============================================================

def sample_details(hash_value):

    return {

        "case_reference":
        "CASE/CYB/2026/0417, Cyber Crime Police Station, Chennai",

        "electronic_record":
        "Forensic image EX-01.dd acquired from laptop, "
        "serial LT-9931-A",

        "source_device":
        "Dell Latitude 5420, serial LT-9931-A, "
        "seized on 20-08-2026 from the accounts section",

        "regular_use_statement":
        "The said computer was used regularly to store "
        "and process information for the ordinary activities "
        "of the organisation during the relevant period.",

        "ordinary_activity":
        "Information of the kind contained in the electronic "
        "record was regularly fed into the computer in the "
        "ordinary course of the said activities.",

        "proper_operation":
        "The computer was operating properly throughout "
        "the material period; where it did not operate "
        "properly, that did not affect the electronic record "
        "or its accuracy.",

        "accuracy_statement":
        "The information contained in the electronic record "
        "reproduces or is derived from information fed into "
        "the computer in the ordinary course of the said "
        "activities.",

        "hash_value":
        hash_value,

        "deponent_name":
        "A. Rao",

        "deponent_designation":
        "Scientific Officer (Cyber Forensics), "
        "State Forensic Science Laboratory",

        "date_of_certificate":
        datetime.now(IST).strftime("%d-%m-%Y")
    }


# ============================================================
# 7. TEST CASES
# ============================================================

def run_tests():

    # Generate SHA-256 for sample electronic evidence
    evidence_data = b"forensic image contents"

    evidence_hash = hashlib.sha256(
        evidence_data
    ).hexdigest()


    good = sample_details(
        evidence_hash
    )


    results = []


    # --------------------------------------------------------
    # TC1 - Complete details accepted
    # --------------------------------------------------------

    certificate, missing = generate_65b_certificate(
        good
    )

    results.append(
        (
            "TC1 complete details accepted",
            missing == []
            and
            "SECTION 65B(4)" in certificate
        )
    )


    # --------------------------------------------------------
    # TC2 - Hash included in certificate
    # --------------------------------------------------------

    results.append(
        (
            "TC2 hash embedded in certificate",
            evidence_hash in certificate
        )
    )


    # --------------------------------------------------------
    # TC3 - Four statutory statements present
    # --------------------------------------------------------

    four_statements_present = all(
        item in certificate
        for item in [
            "(a)",
            "(b)",
            "(c)",
            "(d)"
        ]
    )


    results.append(
        (
            "TC3 four statutory statements present",
            four_statements_present
        )
    )


    # --------------------------------------------------------
    # TC4 - Missing field detected
    # --------------------------------------------------------

    incomplete = dict(good)

    incomplete[
        "deponent_designation"
    ] = ""


    missing_fields = validate(
        incomplete
    )


    results.append(
        (
            "TC4 missing field detected",
            missing_fields
            ==
            ["deponent_designation"]
        )
    )


    # --------------------------------------------------------
    # TC5 - Strict mode refuses incomplete certificate
    # --------------------------------------------------------

    strict_error = False

    try:

        generate_65b_certificate(
            incomplete,
            strict=True
        )

    except CertificateError:

        strict_error = True


    results.append(
        (
            "TC5 strict mode raises on incomplete",
            strict_error
        )
    )


    # --------------------------------------------------------
    # TC6 - Draft mode lists missing fields
    # --------------------------------------------------------

    draft, missing_draft = (
        generate_65b_certificate(
            incomplete,
            strict=False
        )
    )


    results.append(
        (
            "TC6 draft mode lists gaps",
            missing_draft
            ==
            ["deponent_designation"]
            and
            len(draft) > 0
        )
    )


    # --------------------------------------------------------
    # TC7 - Whitespace treated as missing
    # --------------------------------------------------------

    whitespace_data = dict(good)

    whitespace_data[
        "case_reference"
    ] = " "


    whitespace_missing = validate(
        whitespace_data
    )


    results.append(
        (
            "TC7 whitespace treated as missing",
            "case_reference"
            in
            whitespace_missing
        )
    )


    # --------------------------------------------------------
    # TC8 - Empty input detects all fields
    # --------------------------------------------------------

    empty_missing = validate({})


    results.append(
        (
            "TC8 empty input lists all fields",
            len(empty_missing)
            ==
            len(REQUIRED_FIELDS)
        )
    )


    # ========================================================
    # DISPLAY TEST RESULTS
    # ========================================================

    print("=" * 70)
    print("SECTION 65B(4) CERTIFICATE - TEST RESULTS")
    print("=" * 70)


    for name, passed in results:

        print(
            f"{name:<45} -> "
            f"{'PASS' if passed else 'FAIL'}"
        )


    passed_count = sum(
        1
        for _, passed in results
        if passed
    )


    print("-" * 70)

    print(
        f"RESULT: {passed_count}/{len(results)} "
        "test cases passed"
    )

    print("=" * 70)


    return (
        passed_count == len(results)
    )


# ============================================================
# 8. RUN TESTS
# ============================================================

run_tests()


# ============================================================
# 9. GENERATE FINAL CERTIFICATE
# ============================================================

print("\n")
print("=" * 70)
print("GENERATING SECTION 65B(4) CERTIFICATE")
print("=" * 70)


# Create sample evidence
evidence = (
    b"Digital evidence for forensic examination"
)


# Calculate SHA-256
sha256_hash = hashlib.sha256(
    evidence
).hexdigest()


# Create case details
details = sample_details(
    sha256_hash
)


# Validate
missing = validate(
    details
)


if len(missing) == 0:

    certificate, gaps = (
        generate_65b_certificate(
            details,
            strict=True
        )
    )

    print("\n")
    print(certificate)

else:

    print(
        "Certificate cannot be generated."
    )

    print(
        "Missing fields:",
        missing
    )


# ============================================================
# 10. FINAL VALIDATION
# ============================================================

print("\n")
print("=" * 70)
print("CERTIFICATE VALIDATION")
print("=" * 70)


final_missing = validate(
    details
)


if len(final_missing) == 0:

    print(
        "STATUS: VALID"
    )

    print(
        "All mandatory fields are present."
    )

    print(
        "Four Section 65B(2) statements are included."
    )

    print(
        "SHA-256 evidence hash is included."
    )

else:

    print(
        "STATUS: INCOMPLETE"
    )

    print(
        "Missing fields:",
        final_missing
    )


print("=" * 70)
print("EXPERIMENT 4 COMPLETED SUCCESSFULLY")
print("=" * 70)

SECTION 65B(4) CERTIFICATE - TEST RESULTS
TC1 complete details accepted                 -> PASS
TC2 hash embedded in certificate              -> PASS
TC3 four statutory statements present         -> PASS
TC4 missing field detected                    -> PASS
TC5 strict mode raises on incomplete          -> PASS
TC6 draft mode lists gaps                     -> PASS
TC7 whitespace treated as missing             -> PASS
TC8 empty input lists all fields              -> PASS
----------------------------------------------------------------------
RESULT: 8/8 test cases passed


GENERATING SECTION 65B(4) CERTIFICATE


CERTIFICATE UNDER SECTION 65B(4)
OF THE INDIAN EVIDENCE ACT, 1872

Case Reference:
CASE/CYB/2026/0417, Cyber Crime Police Station, Chennai

Date of Certificate:
20-08-2026


1. IDENTIFICATION OF THE ELECTRONIC RECORD
------------------------------------------------------------

Forensic image EX-01.dd acquired from laptop, serial LT-9931-A

SHA-256 Hash Value:
5963d2353743e53cf945